In [0]:

# ============================================================
# BANGLAPAY | PHASE 1
# BATCH LANDING FILE VALIDATION
# Environment: DEV
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql import Row

PG_BATCH_ID = (
    "d13a4d0f-ea63-4946-82f5-79dd05758236"
)

MONGO_BATCH_ID = (
    "5bea1f39-749d-4a53-b599-e0952a2526ee"
)


# ------------------------------------------------------------
# Load extraction manifests
# ------------------------------------------------------------

pg_manifest = (
    spark.table(
        "banglapay.ops.postgres_batch_manifest"
    )
    .filter(F.col("batch_id") == PG_BATCH_ID)
    .collect()
)

mongo_manifest = (
    spark.table(
        "banglapay.ops.mongodb_batch_manifest"
    )
    .filter(F.col("batch_id") == MONGO_BATCH_ID)
    .collect()
)

if len(pg_manifest) != 22:
    raise RuntimeError(
        f"Expected 22 PostgreSQL manifest rows; "
        f"found {len(pg_manifest)}"
    )

if len(mongo_manifest) != 6:
    raise RuntimeError(
        f"Expected 6 MongoDB manifest rows; "
        f"found {len(mongo_manifest)}"
    )


# ------------------------------------------------------------
# Validate actual NDJSON line counts
# ------------------------------------------------------------

results = []


def validate_file(
    source,
    source_object,
    bronze_table,
    expected_rows,
    file_path
):

    expected = int(expected_rows)

    if file_path is None:
        actual = 0
    else:
        actual = (
            spark.read
            .format("text")
            .load(file_path)
            .count()
        )

    status = (
        "PASS"
        if actual == expected
        else "FAIL"
    )

    results.append({

        "source": source,

        "source_object": source_object,

        "bronze_table": bronze_table,

        "manifest_rows": expected,

        "actual_file_lines": actual,

        "status": status,

        "file_path": file_path

    })


# ------------------------------------------------------------
# PostgreSQL files
# ------------------------------------------------------------

for row in pg_manifest:

    validate_file(

        source="PostgreSQL",

        source_object=(
            f"{row['source_schema']}."
            f"{row['source_table']}"
        ),

        bronze_table=row["bronze_table"],

        expected_rows=row["extracted_rows"],

        file_path=row["file_path"]

    )


# ------------------------------------------------------------
# MongoDB files
# ------------------------------------------------------------

for row in mongo_manifest:

    validate_file(

        source="MongoDB",

        source_object=row["source_collection"],

        bronze_table=row["bronze_table"],

        expected_rows=row["extracted_documents"],

        file_path=row["file_path"]

    )


# ------------------------------------------------------------
# Final validation report
# ------------------------------------------------------------

result_df = spark.createDataFrame(
    [Row(**item) for item in results]
)

display(
    result_df.orderBy(
        "source",
        "source_object"
    )
)

passed = sum(
    item["status"] == "PASS"
    for item in results
)

failed = len(results) - passed

print("=" * 60)
print("BANGLAPAY BATCH LANDING VALIDATION")
print("=" * 60)

print("PostgreSQL tables:", len(pg_manifest))
print("MongoDB collections:", len(mongo_manifest))
print("Objects validated:", len(results))
print("Passed:", passed)
print("Failed:", failed)

print(
    "Total manifest rows:",
    sum(item["manifest_rows"] for item in results)
)

print(
    "Total actual file lines:",
    sum(item["actual_file_lines"] for item in results)
)

if failed:
    raise RuntimeError(
        "Batch landing validation failed. "
        "Do not start Auto Loader ingestion."
    )

print("BATCH LANDING VALIDATION: SUCCESS")
